# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [2]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [3]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    op.payment_type,
    op.payment_installments,
    op.payment_sequential,
    oi.price,
    oi.freight_value,
    oi.product_id,
    Count(oi.product_id) AS product_count,                
   
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_payments op 
        ON o.order_id = op.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix,
        o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at, 
        op.payment_value, op.payment_type, op.payment_installments, op.payment_sequential,
        oi.price, oi.freight_value, oi.product_id
    """)
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
0,a3b0fda37bae14cf754877bed475e80c,c9158d089637ab443c78984d20da7fc0,sao paulo,SP,05727,2dd604f5ec1bd2f58c14e9908c7df826,delivered,2018-01-31 16:43:23,2018-02-01 11:35:44,78.70,voucher,1,1,65.90,12.80,6e02baf23db1455640241542c78be139,1
1,38d1cd89306128348ffdf4cc23f3a50a,d491a65a6ef3c04e145d37395996bad7,sao paulo,SP,04548,e3c131bbe953a1cdb1dbaf261715c368,delivered,2017-03-16 16:41:44,2017-03-16 16:41:44,27.71,credit_card,1,1,18.99,8.72,207bb2d8180c2c7654872f6fab96e40e,1
2,314ec9cbe156ddd9cdaa1a9bf021e00e,691d75928efd2c5dd34b678f19783c28,jaboatao dos guararapes,PE,54430,ee4558aeeba185bcd979ed46022947ff,delivered,2018-02-27 10:13:20,2018-02-27 10:30:31,57.62,credit_card,1,1,39.99,17.63,94475071013412139f862c0bd7e3bb37,1
3,1b4ff42fd8acd3dc49e2a137021b3524,39e2788db03e5eaeb2935a7e479682c6,cacapava,SP,12280,9a0ff85473ef691bca39a3636f6753c4,delivered,2018-03-27 19:23:42,2018-03-28 17:30:18,249.94,debit_card,1,1,214.90,35.04,3522a64b7bad6fe22cc046584a9d3fb0,1
4,a4a00c41b5e3ed88bf0e0bb96576c2b2,94e89532396b3c02f648b2bc7f93f528,belford roxo,RJ,26115,4a5cc9b4e332e03d76bf553a7f2fa5d3,delivered,2017-12-15 23:33:32,2017-12-18 00:33:04,176.78,credit_card,3,1,159.90,16.88,c2ccc78b5a924096d5274ca6a3118e47,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107019,e60ef18ae2cc1f0e16b855a4ef2807be,b9badb100ff8ecc16a403111209e3a06,florianopolis,SC,88085,e5283432176f988a5d35d8f6b0711030,delivered,2017-08-25 10:14:02,2017-08-26 02:10:17,507.76,boleto,1,1,69.90,18.53,7340a3839a1de1e99d149b8cf052a2ec,1
107020,5b2b1bdafea16367486b3310068895fd,acea13a79aeaf715ba68122e9deb330a,porto alegre,RS,90870,bcbc83ec3dc71750aff96a593b9e280f,delivered,2018-06-06 11:47:08,2018-06-06 11:55:36,144.54,credit_card,5,1,41.00,31.27,a996a150a7fd0eba6a177ec84b939b51,1
107021,58d9fce5ebd487e6742960be2147af5b,9c34cc0ced21ac538feb7377547b99df,sao paulo,SP,04646,71ba7b3f5aca19e723b21de8b67fc3a3,delivered,2018-01-31 22:52:16,2018-01-31 23:10:04,37.48,credit_card,1,1,199.99,37.49,063d96e7b9190a8da0b8920906578383,1
107022,ea0d5d3cb952f08d3734e1ab8814e4c7,f17b7c86a8eaa3ed4056e4dff553490a,rio claro,SP,13500,e06c6d2cccfbf53339917bf60c317955,delivered,2018-03-11 11:45:43,2018-03-12 17:29:47,87.67,credit_card,4,1,37.50,18.23,0449efdb5b6a5d301b098972abd1749e,2


In [4]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,payment_installments,payment_sequential,price,freight_value,product_count
count,107024,107010,107024.000000,107024.000000,107024.000000,107024.000000,107024.000000,107024.000000
mean,2017-12-30 11:59:56.101313,2017-12-30 23:14:45.274011,157.284282,2.895780,1.093801,124.537146,20.165118,1.098828
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.000000,1.000000,0.850000,0.000000,1.000000
25%,2017-09-10 21:37:30,2017-09-11 17:24:42,57.740000,1.000000,1.000000,40.000000,13.150000,1.000000
50%,2018-01-17 21:44:28.500000,2018-01-18 09:34:13,102.260000,1.000000,1.000000,78.000000,16.350000,1.000000
75%,2018-05-03 23:51:46,2018-05-04 13:33:08,176.160000,4.000000,1.000000,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,24.000000,29.000000,6735.000000,409.680000,20.000000
std,NaN,NaN,218.601318,2.723853,0.718380,189.781733,15.950996,0.452583


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [5]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
payment_value                      float64
payment_type                        object
payment_installments                 int64
payment_sequential                   int64
price                              float64
freight_value                      float64
product_id                          object
product_count                        int64
dtype: object

In [6]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'payment_type': 'category', 
              'payment_installments': 'int16',
              'payment_sequential': 'int16',
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category',
               'product_count': 'int16'}
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
payment_value                     float32
payment_type                     category
payment_installments                int16
payment_sequential                  int16
price                             float32
freight_value                     float32
product_id                       category
product_count                       int16
dtype: object

In [7]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,payment_installments,payment_sequential,price,freight_value,product_count
count,107024,107010,107024.000000,107024.000000,107024.000000,107024.000000,107024.000000,107024.000000
mean,2017-12-30 11:59:56,2017-12-30 23:14:45,157.284271,2.895780,1.093801,124.537148,20.165119,1.098828
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.000000,1.000000,0.850000,0.000000,1.000000
25%,2017-09-10 21:37:30,2017-09-11 17:24:42,57.740002,1.000000,1.000000,40.000000,13.150000,1.000000
50%,2018-01-17 21:44:28,2018-01-18 09:34:13,102.260002,1.000000,1.000000,78.000000,16.350000,1.000000
75%,2018-05-03 23:51:46,2018-05-04 13:33:08,176.160004,4.000000,1.000000,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080078,24.000000,29.000000,6735.000000,409.679993,20.000000
std,NaN,NaN,218.601318,2.723853,0.718380,189.781738,15.950995,0.452583


In [8]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           14
payment_value                0
payment_type                 0
payment_installments         0
payment_sequential           0
price                        0
freight_value                0
product_id                   0
product_count                0
dtype: int64

In [9]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
2511,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,106.809998,boleto,1,1,79.989998,26.820000,c6dd917a0be2a704582055949915ab32,1
2839,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,127.040001,boleto,1,1,49.000000,14.520000,8c5876b1c7768217964f353bc7e64393,2
7081,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,95.760002,boleto,1,1,79.989998,15.770000,c6dd917a0be2a704582055949915ab32,1
12045,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,77.059998,boleto,1,1,59.900002,17.160000,7868a64aa111bbb4f41f8e1146c0becb,1
14226,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,154.229996,boleto,1,1,135.000000,19.230000,4fd676d9c4723d475026e40aeae56957,1
15714,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,54.509998,boleto,1,1,39.990002,14.520000,5ab02ca028398131a5ae91401eb49788,1
21457,29c35fc91fc13fb5073c8f30505d860d,7e1a5ca61b572d76b64b6688b9f96473,caninde,CE,62700,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,106.809998,boleto,1,1,79.989998,26.820000,c6dd917a0be2a704582055949915ab32,1
22925,4c1ccc74e00993733742a3c786dc3c1f,91efb7fcabc17925099dced52435837f,novo hamburgo,RS,93548,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,396.859985,boleto,1,1,379.000000,17.860001,2c2b6a28924791234bd386bddb17512e,1
28891,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,163.429993,boleto,1,1,149.800003,13.630000,cae2e38942c8489d9d7a87a3f525c06b,1
32543,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,157.190002,boleto,1,1,133.990005,23.200001,db8ed3d08891d16a2438a67ab3acb740,1


In [10]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,2,0
canceled,489,0
delivered,104669,14
invoiced,333,0
processing,324,0
shipped,1186,0
unavailable,7,0


In [11]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count


In [12]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 0


In [13]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [14]:
test['sum_status']=test.sum(axis=1)

In [15]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
0016dfedd97fc2950e388d2971d718c7,0,0,2,0,0,0,0,2
002f19a65a2ddd70a090297872e6d64e,0,0,0,0,0,2,0,2
002f98c0f7efd42638ed6100ca699b42,0,0,2,0,0,0,0,2
00337fe25a3780b3424d9ad7c5a4b35e,0,0,2,0,0,0,0,2
005d9a5423d47281ac463a968b3936fb,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffb8f7de8940249a3221252818937ecb,0,0,3,0,0,0,0,3
ffb9a9cd00c74c11c24aa30b3d78e03b,0,0,3,0,0,0,0,3
ffc16cecff8dc037f60458f28d1c1ba5,0,0,2,0,0,0,0,2


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [16]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [17]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False) # type: ignore

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
895ab968e7bb0d5659d16cd74cd1650c,0,0,42,0,0,0,0,42
fa65dad1b0e818e3ccc5cb0e39231352,0,0,0,0,0,29,0,29
ccf804e764ed5650cd8759557269dc13,0,0,26,0,0,0,0,26
68986e4324f6a21481df4e6e89abcf01,0,0,24,0,0,0,0,24
285c2e15bebd4ac83635ccc563dc71f4,0,0,22,0,0,0,0,22
...,...,...,...,...,...,...,...,...
5b33495590a02d7e9ce6fbdc52f143d9,0,0,2,0,0,0,0,2
5b33244d158adb23abb0d481fa052af8,0,0,2,0,0,0,0,2
5b2c9910c96e7dec4dac98d009a43858,0,0,2,0,0,0,0,2


## Filterung nach der Bestellung mit den meisten Duplikaten

In [18]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'
product_id = 'ebf9bc6cd600eadd681384e3116fda85'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[(df_rfm_eda['order_id'] == order_id) & (df_rfm_eda['product_id'] == product_id)]

order_data.head(63).sort_values('payment_sequential', ascending=True)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
50782,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,1,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
74267,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,2,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
9006,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,3,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
69151,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,4,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
27380,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,5,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
15596,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,6,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
84414,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.600000,voucher,1,7,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
4144,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,0.410000,voucher,1,8,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
49137,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,9,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
12264,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,10,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2


In [19]:
print(order_data['payment_value'].sum())  # Oder gefiltert.sum(numeric_only=True) für alle numerischen
print(2*12.99+23.20999)

161.32
49.18999


In [20]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'
# product_id = 'ebf9bc6cd600eadd681384e3116fda85'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[(df_rfm_eda['order_id'] == order_id)] # & (df_rfm_eda['product_id'] == product_id)]

order_data.head(63).sort_values('payment_sequential', ascending=True)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
453,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,309.000000,1.84,1065e0ebef073787a7bf691924c60eeb,1
4154,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,159.000000,3.67,0de59eddc63167215c972b0d785ffa7b,2
5820,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,109.900002,0.15,4a5c3967bfd3629fe07ef4d0cc8c3818,1
7435,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,33.900002,1.84,0cf2faf9749f53924cea652a09d8e327,1
19036,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,56.000000,3.68,309dd69eb83cea38c51709d62befe1a4,2
75908,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,63.700001,0.15,21b524c4c060169fa75ccf08c7da4627,1
77515,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,95.900002,0.15,5dae498eff2d80057f56122235a36aff,1
82692,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,1157.280029,credit_card,10,1,95.900002,0.15,678c229b41c0e497d35a25a8be1cc631,1


In [21]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'
product_id = 'ebf9bc6cd600eadd681384e3116fda85'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[(df_rfm_eda['order_id'] == order_id) & (df_rfm_eda['product_id'] == product_id)]

order_data.head(63).sort_values('payment_sequential', ascending=True)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
50782,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,1,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
74267,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,2,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
9006,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,3,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
69151,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,4,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
27380,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,5,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
15596,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,6,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
84414,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.600000,voucher,1,7,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
4144,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,0.410000,voucher,1,8,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
49137,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,9,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2
12264,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,10,12.99,23.209999,ebf9bc6cd600eadd681384e3116fda85,2


## Erkenntnis
Es handelt sich hier um keine Duplikate. 
Dadurch das die Payment Tabelle mit integriert wurde sieht es so als würden hier sehr viele Duplikate bestehen.
Allerding, hat er wie man oben sieht, ein Produkt 2 mal gekauft in einer Bestellung und diese in 21 Raten mit Gutschein bezahlt

Damit haben wir keine Duplikate im Datensatz

In [22]:
df_payment = sql("""
SELECT *
FROM order_payments
    """)
test5 = df_payment[df_payment['payment_type'] == 'credit_card']
test5.sort_values(by='payment_sequential', ascending=False).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82
74423,86dda1172108d1601101914c45b51699,2,credit_card,8,300.80
95964,f464a113cdd89833ee98d4a56f41d8c5,2,credit_card,1,20.24
53071,dbbb8658d9a27d2be4cefb9a9a5d9e33,2,credit_card,1,40.14
50388,dabb5a87a6d9cc1388abf76cdcf76e5b,2,credit_card,2,180.00
37443,e003d94282e5f1e45760dde794a8a173,2,credit_card,7,75.79
72388,40edd29fcc7f120710d42d0400bc8e29,2,credit_card,2,150.00
72391,0b7ed5e5b6354118fdeec73b9f9fd8a9,2,credit_card,3,161.54
63497,5df5b6ce88b311095393f226e4366bcf,2,credit_card,8,87.51
76348,df68ea0d1a2a2b9d4dfa236820acbe9c,2,credit_card,8,194.44


In [23]:
test5[test5['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82


In [24]:
df_payment[df_payment['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82
36471,a079628ac8002126e75f86b0f87332e4,2,debit_card,1,50.00


In [25]:
df_payment[df_payment['order_id'] == 'b81ef226f3fe1789b1e8b2acac839d17']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33


In [26]:
testvoucher = df_payment[df_payment['payment_type'] == 'voucher']
testvoucher.sort_values(by='payment_sequential', ascending=False).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26
39111,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.05
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
79587,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.27
32393,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.10
39132,ccf804e764ed5650cd8759557269dc13,25,voucher,1,1.53
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
99213,fa65dad1b0e818e3ccc5cb0e39231352,24,voucher,1,0.42
51816,ccf804e764ed5650cd8759557269dc13,24,voucher,1,2.79
60241,ccf804e764ed5650cd8759557269dc13,23,voucher,1,1.03


In [27]:
testvoucher[testvoucher['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c'].sort_values(by='payment_sequential', ascending=True)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
9319,895ab968e7bb0d5659d16cd74cd1650c,1,voucher,1,2.61
52639,895ab968e7bb0d5659d16cd74cd1650c,2,voucher,1,2.61
68148,895ab968e7bb0d5659d16cd74cd1650c,3,voucher,1,2.61
41528,895ab968e7bb0d5659d16cd74cd1650c,4,voucher,1,2.61
72756,895ab968e7bb0d5659d16cd74cd1650c,5,voucher,1,2.61
89944,895ab968e7bb0d5659d16cd74cd1650c,6,voucher,1,2.61
76975,895ab968e7bb0d5659d16cd74cd1650c,7,voucher,1,2.60
58817,895ab968e7bb0d5659d16cd74cd1650c,8,voucher,1,0.41
87611,895ab968e7bb0d5659d16cd74cd1650c,9,voucher,1,2.61
75908,895ab968e7bb0d5659d16cd74cd1650c,10,voucher,1,16.70


In [28]:
df_payment[df_payment['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c'].sort_values(by='payment_sequential', ascending=True)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
9319,895ab968e7bb0d5659d16cd74cd1650c,1,voucher,1,2.61
52639,895ab968e7bb0d5659d16cd74cd1650c,2,voucher,1,2.61
68148,895ab968e7bb0d5659d16cd74cd1650c,3,voucher,1,2.61
41528,895ab968e7bb0d5659d16cd74cd1650c,4,voucher,1,2.61
72756,895ab968e7bb0d5659d16cd74cd1650c,5,voucher,1,2.61
89944,895ab968e7bb0d5659d16cd74cd1650c,6,voucher,1,2.61
76975,895ab968e7bb0d5659d16cd74cd1650c,7,voucher,1,2.60
58817,895ab968e7bb0d5659d16cd74cd1650c,8,voucher,1,0.41
87611,895ab968e7bb0d5659d16cd74cd1650c,9,voucher,1,2.61
75908,895ab968e7bb0d5659d16cd74cd1650c,10,voucher,1,16.70


In [29]:
order_id = 'a079628ac8002126e75f86b0f87332e4'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)



,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count
4151,62d561a5b1260c4476fd985dcae3eb78,369cf098450e6de108ffa64f68b713a1,capanema,PA,68700,a079628ac8002126e75f86b0f87332e4,delivered,2018-04-24 10:19:45,2018-04-24 18:29:37,50.00,debit_card,1,2,99.989998,52.830002,0983cd4a5cabf1099659ce461511963c,1
53979,62d561a5b1260c4476fd985dcae3eb78,369cf098450e6de108ffa64f68b713a1,capanema,PA,68700,a079628ac8002126e75f86b0f87332e4,delivered,2018-04-24 10:19:45,2018-04-24 18:29:37,102.82,credit_card,3,3,99.989998,52.830002,0983cd4a5cabf1099659ce461511963c,1


In [30]:
order_data.groupby('order_id')['payment_value'].nunique().sort_values(ascending=False)

/var/folders/4y/n75k9x_d6nx_z4j3d1jd_z080000gn/T/ipykernel_77951/4152580056.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  order_data.groupby('order_id')['payment_value'].nunique().sort_values(ascending=False)


order_id
a079628ac8002126e75f86b0f87332e4    2
00010242fe8c5a6d1ba2dd792cb16214    0
ab36f41e5d298012acb2179337f1a9db    0
ab3a669dbdc7a8d447bb6fc7a158cf94    0
ab3a6663f21c1a36401300460d3c41cd    0
                                   ..
5561adcb0fd46da4cad3048fa4e7fc00    0
555e60e282181725debc9eb2d69fda3f    0
555e4d40fb6beea866d46eb6a5a01b41    0
555e1afa0cf180760b7ea9f6d8ebc329    0
fffe41c64501cc87c801fd61db3f6244    0
Name: payment_value, Length: 98665, dtype: int64

In [31]:
pd.crosstab(order_data['payment_value'], order_data['order_id'])

order_id,a079628ac8002126e75f86b0f87332e4
payment_value,
50.00,1
102.82,1


In [32]:
normancode = sql("""
SELECT *
FROM order_payments
WHERE order_id = 'a079628ac8002126e75f86b0f87332e4'
ORDER BY payment_sequential;
                                  """)
normancode

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,a079628ac8002126e75f86b0f87332e4,2,debit_card,1,50.00
1,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82


In [33]:
# NaNs nur in order_approved_at
df_rfm_eda[df_rfm_eda['order_approved_at'].isna()].head(10)


,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id,product_count


### EDA für zweite Kernaufgabe

In [34]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    r.review_score
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
JOIN order_items oi ON o.order_id = oi.order_id  
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
    """)

In [35]:
df_pc_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,product_id,product_category_name_english,review_score
0,73fc7af87114b39712e6da79b0a377eb,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,397.26,185.00,13.63,fd25ab760bfbba13c198fa3b4f1a0cd3,sports_leisure,4
1,a548910a1c6147796b98fdf73dbeba33,delivered,2018-02-28 12:25:19,2018-02-28 12:48:39,88.09,79.79,8.30,be0dbdc3d67d55727a65d4cd696ca73c,computers_accessories,5
2,f9e4b658b201a9f2ecdecbb34bed034b,delivered,2018-02-03 09:56:22,2018-02-03 10:33:41,194.12,149.00,45.12,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,5
3,658677c97b385a9be170737859d3511b,delivered,2017-04-09 17:41:13,2017-04-09 17:55:19,222.84,179.99,42.85,52c80cedd4e90108bf4fa6a206ef6b03,garden_tools,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,delivered,2018-02-10 10:59:03,2018-02-10 15:48:21,1333.25,1199.00,134.25,3880d25d502b15b1de6fddc42ad1d67a,sports_leisure,5
...,...,...,...,...,...,...,...,...,...,...
116008,b538efecd9ba6d45acff500110ce1e45,delivered,2018-01-24 21:03:02,2018-01-24 21:18:20,33.59,18.49,15.10,8db8c6b5b338e7a8d3332037c2c218f7,telephony,<NA>
116009,9757bb9b0d295ca539b1b15c3eab9b2e,delivered,2017-07-30 20:17:31,2017-07-30 20:30:06,231.27,205.00,26.27,f1c7f353075ce59d8a6f3cf58f419c9c,bed_bath_table,<NA>
116010,8fccc7922426ac1c14efd13857e57bff,delivered,2017-03-19 17:02:45,2017-03-19 17:02:45,271.90,119.90,16.05,e6b6e72a0e6be244a69261788f086429,bed_bath_table,<NA>
116011,8fccc7922426ac1c14efd13857e57bff,delivered,2017-03-19 17:02:45,2017-03-19 17:02:45,271.90,119.90,16.05,e6b6e72a0e6be244a69261788f086429,bed_bath_table,<NA>


In [36]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,review_score
count,116013,115999,116013.000000,116013.000000,116013.000000,115066.0
mean,2017-12-31 04:53:31.985683,2017-12-31 16:18:46.135906,172.438748,120.449621,20.062503,4.045913
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000,1.0
25%,2017-09-12 13:49:33,2017-09-12 21:02:44.500000,61.000000,39.900000,13.080000,4.0
50%,2018-01-18 20:49:36,2018-01-19 10:55:40,108.120000,74.900000,16.320000,5.0
75%,2018-05-04 13:28:13,2018-05-04 19:35:16,189.560000,134.200000,21.220000,5.0
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,6735.000000,409.680000,5.0
std,NaN,NaN,266.087022,182.709965,15.834814,1.376048


In [37]:
df_pc_eda.dtypes

order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
payment_value                           float64
price                                   float64
freight_value                           float64
product_id                               object
product_category_name_english            object
review_score                              Int64
dtype: object

In [38]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category', 
              'product_category_name_english': 'category',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

order_id                              category
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
payment_value                          float32
price                                  float32
freight_value                          float32
product_id                            category
product_category_name_english         category
review_score                          category
dtype: object

In [39]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value
count,116013,115999,116013.000000,116013.000000,116013.000000
mean,2017-12-31 04:53:31,2017-12-31 16:18:46,172.438751,120.449623,20.062504
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000
25%,2017-09-12 13:49:33,2017-09-12 21:02:44,61.000000,39.900002,13.080000
50%,2018-01-18 20:49:36,2018-01-19 10:55:40,108.120003,74.900002,16.320000
75%,2018-05-04 13:28:13,2018-05-04 19:35:16,189.559998,134.199997,21.219999
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080078,6735.000000,409.679993
std,NaN,NaN,266.087036,182.709961,15.834814


In [40]:
df_pc_eda.isna().sum()

order_id                           0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
payment_value                      0
price                              0
freight_value                      0
product_id                         0
product_category_name_english      0
review_score                     947
dtype: int64

In [41]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 11410


In [42]:
test2 = pd.crosstab(df_pc_eda['order_status'], df_pc_eda['order_id']).T
test2.head(10)

order_status,approved,delivered,invoiced,processing,shipped
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,1,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,1,0,0,0
000229ec398224ef6ca0657da4fc703e,0,1,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,1,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,1,0,0,0
00048cc3ae777c65dbb7d2a0634bc1ea,0,1,0,0,0
00054e8431b9d7675808bcb819fb4a32,0,1,0,0,0
000576fe39319847cbb9d288c5617fa6,0,1,0,0,0
0005a1a1728c9d785b8e2b08b904576c,0,1,0,0,0


In [43]:
test2['sum_status']=test2.sum(axis=1)
test2.loc[test2['sum_status']!=1, :] 

order_status,approved,delivered,invoiced,processing,shipped,sum_status
order_id,,,,,,
0008288aa423d2a3f00fcb17cd7d8719,0,2,0,0,0,2
00143d0f86d6fbd9f9b38ab440ac16f5,0,3,0,0,0,3
0016dfedd97fc2950e388d2971d718c7,0,2,0,0,0,2
001ab0a7578dd66cd4b0a71f5b6e1e41,0,3,0,0,0,3
001d8f0e34a38c37f7dba2a37d4eba8b,0,2,0,0,0,2
...,...,...,...,...,...,...
ffd84ab39cd5e873d8dba24342e65c01,0,2,0,0,0,2
ffe4b41e99d39f0b837a239110260530,0,2,0,0,0,2
ffecd5a79a0084f6a592288c67e3c298,0,3,0,0,0,3


In [44]:
test2.loc[test2['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,delivered,invoiced,processing,shipped,sum_status
order_id,,,,,,
895ab968e7bb0d5659d16cd74cd1650c,0,63,0,0,0,63
fedcd9f7ccdc8cba3a18defedd1a5547,0,38,0,0,0,38
fa65dad1b0e818e3ccc5cb0e39231352,0,0,0,0,29,29
ccf804e764ed5650cd8759557269dc13,0,26,0,0,0,26
c6492b842ac190db807c15aff21a7dd6,0,24,0,0,0,24
...,...,...,...,...,...,...
6cdd625fb7db568f0f52136d0a8374a3,0,2,0,0,0,2
6ce409ae9fc35c1c78c3190b660a23b4,0,2,0,0,0,2
6ceabf34d230c31f161988dd2ff8fa92,0,2,0,0,0,2


### EDA für dritte Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,             
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [46]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,seller_city,seller_state,seller_lat,seller_lng,customer_city,customer_state,product_category_name_english
0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,-23.494316,-46.364539,franca,SP,office_furniture
1,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,5,b8bc237ba3788b23da09c0f1f3a3288c,itajai,SC,-26.913214,-48.677675,sao bernardo do campo,SP,housewares
2,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,-23.494316,-46.364539,sao paulo,SP,office_furniture
3,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,-23.494316,-46.364539,mogi das cruzes,SP,office_furniture
4,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,5,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,-21.754867,-48.838906,campinas,SP,home_confort
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16008317,497557705ed42b9e7678b808d1f21dbb,delivered,2017-09-08 15:00:33,2017-09-08 15:10:22,2017-09-11 23:34:56,2017-09-19 20:58:46,2017-09-29,5,d379f449f2a3b271bc01c0782020f705,divinopolis,MG,-20.145870,-44.894220,rio de janeiro,RJ,housewares
16008318,c44fcdcfefce1eafc0ca64ca3d6af678,delivered,2018-02-02 14:50:36,2018-02-02 15:18:03,2018-02-03 00:42:27,2018-02-08 18:22:26,2018-02-22,5,2b1a40c1daabc6ca280c4b815c101841,divinopolis,MG,-20.146235,-44.889731,belo horizonte,MG,auto
16008319,497557705ed42b9e7678b808d1f21dbb,delivered,2017-09-08 15:00:33,2017-09-08 15:10:22,2017-09-11 23:34:56,2017-09-19 20:58:46,2017-09-29,5,d379f449f2a3b271bc01c0782020f705,divinopolis,MG,-20.146235,-44.889731,rio de janeiro,RJ,housewares
16008320,c44fcdcfefce1eafc0ca64ca3d6af678,delivered,2018-02-02 14:50:36,2018-02-02 15:18:03,2018-02-03 00:42:27,2018-02-08 18:22:26,2018-02-22,5,2b1a40c1daabc6ca280c4b815c101841,divinopolis,MG,-20.146615,-44.892592,belo horizonte,MG,auto


In [ ]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score
count,110750,110736,109605,108457,110750,110750.000000
mean,2018-01-01 12:44:37.691458,2018-01-02 00:18:39.936452,2018-01-05 14:22:58.117586,2018-01-15 00:21:00.601399,2018-01-25 08:59:40.301580,4.035395
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000
25%,2017-09-14 08:05:20,2017-09-14 13:30:22,2017-09-18 22:27:28,2017-09-26 23:18:38,2017-10-05 00:00:00,4.000000
50%,2018-01-20 22:39:33,2018-01-22 13:53:44,2018-01-24 23:32:35,2018-02-03 18:38:41,2018-02-16 00:00:00,5.000000
75%,2018-05-05 15:14:23.750000,2018-05-05 22:13:43,2018-05-08 15:07:00,2018-05-16 12:53:08,2018-05-28 00:00:00,5.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000
std,NaN,NaN,NaN,NaN,NaN,1.385325


In [ ]:
df_service_eda.dtypes


order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_score                              int64
seller_id                                object
seller_city                              object
seller_state                             object
customer_city                            object
customer_state                           object
product_category_name_english            object
dtype: object

In [ ]:
df_service_eda.isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date     1145
order_delivered_customer_date    2293
order_estimated_delivery_date       0
review_score                        0
seller_id                           0
seller_city                         0
seller_state                        0
customer_city                       0
customer_state                      0
product_category_name_english       0
dtype: int64